In [ ]:
# -*- coding: utf-8 -*-
"""AGROIA - EXTENSIVOS v2.5
Automatically generated by Colab.
Original file is located at
    https://colab.research.google.com/drive/1ImnuhqhXccAKT-58xAxQrbNPA9KBK4YB
"""

!pip install earthengine-api geopandas reportlab pypdf matplotlib scikit-learn requests folium -q

# -*- coding: utf-8 -*-
"""
AGROIA - MOTOR UNIFICADO V2.5
Gestión de Riesgo en Cultivos Extensivos — Zona Núcleo / Córdoba
Novedad v2.5: umbral_clima por cultivo, nombre_lote unificado, outputs consolidados.
"""

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import folium
from folium import plugins
from datetime import datetime
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

import ee

# ============================================================================
# 0. AUTENTICACIÓN GEE
# ============================================================================

def init_gee(project_id=None):
    try:
        if project_id:
            ee.Initialize(project=project_id)
        else:
            ee.Initialize()
        print("✓ GEE inicializado correctamente.")
    except Exception:
        print("→ Autenticando GEE por primera vez...")
        ee.Authenticate()
        if project_id:
            ee.Initialize(project=project_id)
        else:
            ee.Initialize()
        print("✓ GEE autenticado e inicializado.")


# ============================================================================
# 1. CONFIGURACIÓN MAESTRA Y MATRIZ FENOLÓGICA
# ============================================================================

CONFIG = {
    "maiz": {
        "tbase":         10,
        "umbral_calor":  35,   # °C — umbral temperatura para contar horas
        "umbral_clima":  40,   # h  — horas máximas para score clima = 0
        "mes_critico":   1,
        "pesos": {10: 0.2, 11: 0.5, 12: 1.0, 1: 1.0, 2: 0.6, 3: 0.2},
        "color":         "#2D6A4F",
        "biblio":        "INTA Marcos Juárez / Univ. Nebraska",
        "ndvi_min":      0.25,
        "ndvi_max":      0.92,
    },
    "soja": {
        "tbase":         10,
        "umbral_calor":  35,
        "umbral_clima":  35,   # h — floración más sensible
        "mes_critico":   2,
        "pesos": {11: 0.3, 12: 0.7, 1: 1.0, 2: 1.0, 3: 0.5, 4: 0.2},
        "color":         "#40916C",
        "biblio":        "INTA Marcos Juárez",
        "ndvi_min":      0.25,
        "ndvi_max":      0.90,
    },
    "trigo": {
        "tbase":         0,
        "umbral_calor":  30,
        "umbral_clima":  30,   # h — espigazón septiembre más sensible
        "mes_critico":   10,
        "pesos": {6: 0.1, 7: 0.2, 8: 0.4, 9: 1.0, 10: 1.0, 11: 0.5},
        "color":         "#C29B0C",
        "biblio":        "INTA Pergamino",
        "ndvi_min":      0.20,
        "ndvi_max":      0.88,
    },
}

# ============================================================================
# 2. MÓDULO CLIMÁTICO — NASA POWER + SINUSOIDAL
# ============================================================================

def estimate_hours_over_threshold(tmax, tmin, umbral):
    if tmax <= umbral:
        return 0.0
    if tmin >= umbral:
        return 24.0
    tmean = (tmax + tmin) / 2.0
    tamp  = (tmax - tmin) / 2.0
    x = np.clip((umbral - tmean) / tamp, -1.0, 1.0)
    horas = (24.0 / np.pi) * np.arccos(x)
    return round(horas, 2)

def get_nasa_climate_safe(lat, lon, year, cultivo_conf):
    mes    = cultivo_conf['mes_critico']
    umbral = cultivo_conf['umbral_calor']
    end_month = "0430" if mes <= 6 else "1130"
    start = f"{year}0101"
    end   = f"{year}{end_month}"
    url = (f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M_MAX,T2M_MIN&community=AG"
           f"&longitude={lon}&latitude={lat}&start={start}&end={end}&format=JSON")
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()
        tmax_d = data['properties']['parameter']['T2M_MAX']
        tmin_d = data['properties']['parameter']['T2M_MIN']
        df = pd.DataFrame({'date': pd.to_datetime(list(tmax_d.keys()), format='%Y%m%d'),
                           'Tmax': list(tmax_d.values()), 'Tmin': list(tmin_d.values())})
        df = df[(df['Tmax'] > -900) & (df['Tmin'] > -900)].copy()
        df_mes = df[df['date'].dt.month == mes]
        if df_mes.empty: return 0.0, 'no_data'
        df_mes['horas'] = df_mes.apply(lambda row: estimate_hours_over_threshold(row['Tmax'], row['Tmin'], umbral), axis=1)
        return round(df_mes['horas'].sum(), 2), 'ok'
    except Exception as e:
        return 0.0, f'error: {e}'

# ============================================================================
# 3. MÓDULO SATELITAL — GEE + CLOUD MASKING
# ============================================================================

def get_gee_ndvi(geom_ee, year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, 'month')
    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(geom_ee).filterDate(start, end)
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
                  .map(lambda img: img.normalizedDifference(['B8', 'B4']).rename('ndvi')))
    if collection.size().getInfo() == 0: return None
    return collection.median().reduceRegion(reducer=ee.Reducer.mean(), geometry=geom_ee, scale=10, maxPixels=1e9).get('ndvi').getInfo()


def get_gee_ndvi_ventana(geom_ee, year, month, max_delta=2):
    for delta in range(0, max_delta + 1):
        for signo in ([0] if delta == 0 else [1, -1]):
            mes_intento = month + (delta * signo)
            if mes_intento < 1 or mes_intento > 12:
                continue
            val = get_gee_ndvi(geom_ee, year, mes_intento)
            if val is not None:
                aviso = None if delta == 0 else (
                    f"⚠ {year}: mes critico ({month:02d}) sin imagenes limpias. "
                    f"NDVI tomado de mes {mes_intento:02d} (delta {delta} mes). "
                    f"Aplicar con criterio agronomico."
                )
                return val, mes_intento, aviso
    return None, None, (
        f"Sin imagenes validas en ventana +/-{max_delta} meses "
        f"alrededor del mes {month:02d} para {year}. Ano excluido del Score."
    )


def validar_shapefile(shp_path):
    from shapely.validation import explain_validity
    warnings_list = []
    try:
        lote = gpd.read_file(shp_path)
    except Exception as e:
        raise ValueError("No se pudo leer el shapefile.\n   Error: " + str(e) +
                         "\n   Verificar que .shp, .shx, .dbf y .prj esten completos.")
    if lote.empty:
        raise ValueError("El shapefile esta vacio. No contiene geometrias.")
    if lote.crs is None:
        raise ValueError("El shapefile no tiene CRS definido.\n   Falta el archivo .prj.")
    if len(lote) > 1:
        warnings_list.append(f"El shapefile tiene {len(lote)} poligonos. Se analizara solo el de mayor area.")
        lote = lote.to_crs("EPSG:4326")
        lote["_area"] = lote.geometry.area
        lote = lote.sort_values("_area", ascending=False).head(1).drop(columns="_area").reset_index(drop=True)
    if lote.crs.to_epsg() != 4326:
        lote = lote.to_crs("EPSG:4326")
    geom = lote.geometry.iloc[0]
    if geom.geom_type == "MultiPolygon":
        warnings_list.append("MultiPolygon detectado. Se usara el poligono de mayor area.")
        geom = max(geom.geoms, key=lambda p: p.area)
        lote.at[0, "geometry"] = geom
    elif geom.geom_type != "Polygon":
        raise ValueError("Tipo de geometria no soportado: " + geom.geom_type)
    if not geom.is_valid:
        motivo = explain_validity(geom)
        warnings_list.append("Geometria con errores topologicos: " + motivo + ". Corrigiendo con buffer 0.")
        lote.at[0, "geometry"] = geom.buffer(0)
        geom = lote.geometry.iloc[0]
    zona_utm   = int((geom.centroid.x + 180) / 6) + 1
    hemisferio = 32700 if geom.centroid.y < 0 else 32600
    epsg_utm   = hemisferio + zona_utm
    lote_utm   = lote.to_crs(f"EPSG:{epsg_utm}")
    hectareas  = lote_utm.geometry.area.sum() / 10_000
    if hectareas < 1:
        raise ValueError(f"Superficie demasiado pequena: {hectareas:.2f} ha. Minimo 1 ha.")
    if hectareas > 10_000:
        warnings_list.append(f"Superficie muy grande: {hectareas:.0f} ha. GEE puede dar timeout.")
    bounds = geom.bounds
    if not (-90 <= bounds[1] <= 90 and -180 <= bounds[0] <= 180):
        raise ValueError(f"Coordenadas fuera de rango geografico: {bounds}.")
    print(f"✓ Shapefile valido: {hectareas:.2f} ha | {geom.geom_type} | EPSG:{epsg_utm}")
    for w in warnings_list:
        print(f"  ⚠ {w}")
    return lote, warnings_list

def calcular_cv_gee(geom_ee, year, month, scale=20):
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, 'month')
    img = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(geom_ee).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
           .map(lambda i: i.normalizedDifference(['B8', 'B4']).rename('ndvi')).median())
    stats = img.reduceRegion(reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
                             geometry=geom_ee, scale=scale, maxPixels=1e9).getInfo()
    mean, std = stats.get('ndvi_mean'), stats.get('ndvi_stdDev')
    if mean and mean > 0 and std is not None: return round(std / mean, 4), round(mean, 4), round(std, 4)
    return None, None, None

# ============================================================================
# 4. MOTOR DE AMBIENTES (SPATIAL CLUSTERING)
# ============================================================================

def zonificar_lote_gee(geom_ee, year, month, n_clusters=3):
    start = ee.Date.fromYMD(year, month, 1)
    img = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')\
            .filterBounds(geom_ee).filterDate(start, start.advance(1, 'month'))\
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))\
            .median().normalizedDifference(['B8', 'B4']).rename('ndvi')
    sample = img.sample(region=geom_ee, scale=10, numPixels=2000, geometries=True).getInfo()
    if not sample['features']: return None
    coords = [f['geometry']['coordinates'] for f in sample['features']]
    values = [[f['properties']['ndvi']] for f in sample['features']]
    kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(values)
    centers = kmeans.cluster_centers_.flatten()
    order = np.argsort(centers)
    label_map = {order[0]: 'C', order[1]: 'B', order[2]: 'A'}
    zonas_puntos = gpd.GeoDataFrame({
        'zona': [label_map[l] for l in kmeans.labels_],
        'ndvi': [v[0] for v in values]
    }, geometry=gpd.points_from_xy([c[0] for c in coords], [c[1] for c in coords]), crs="EPSG:4326")
    return zonas_puntos

# ============================================================================
# 5. VALIDACIÓN Y SCORE
# ============================================================================

def validar_ndvi(ndvi_val, cultivo, year, mes):
    conf = CONFIG.get(cultivo, CONFIG['maiz'])
    if ndvi_val is None or (isinstance(ndvi_val, float) and np.isnan(ndvi_val)):
        return 'nulo', f"⚠ {year}/{mes:02d}: NDVI nulo."
    if ndvi_val < conf['ndvi_min']:
        return 'sospechoso_bajo', f"⚠ {year}/{mes:02d}: NDVI={ndvi_val:.3f} bajo mínimo."
    if ndvi_val > conf['ndvi_max']:
        return 'sospechoso_alto', f"⚠ {year}/{mes:02d}: NDVI={ndvi_val:.3f} muy alto."
    return 'ok', f"✓ {year}/{mes:02d}: NDVI={ndvi_val:.3f} plausible."

def get_gee_ndvi_validado(geom_ee, year, month, cultivo):
    try:
        val = get_gee_ndvi(geom_ee, year, month)
        status, msg = validar_ndvi(val, cultivo, year, month)
        if status in ('sospechoso_bajo', 'nulo'): return None, status, msg
        return val, status, msg
    except Exception as e: return None, 'error', f"Error GEE: {e}"

def calcular_score(ndvi_critico, horas_calor, ndvi_historico, umbral_clima=40):
    """
    umbral_clima: horas máximas sobre umbral térmico para penalización total.
      maíz=40h | soja=35h | trigo=30h (ref: INTA Marcos Juárez / Nebraska)
    """
    vigor = np.clip(ndvi_critico / 0.9, 0.0, 1.0) * 40.0
    if len(ndvi_historico) >= 3:
        arr = np.array(ndvi_historico)
        cv  = np.std(arr) / np.mean(arr) if np.mean(arr) > 0 else 1.0
        estabilidad = np.clip(1.0 - cv / 0.45, 0.0, 1.0) * 30.0
    else: estabilidad = 15.0
    if len(ndvi_historico) >= 4:
        iso = IsolationForest(contamination=0.2, random_state=42)
        labels = iso.fit_predict(np.array(ndvi_historico).reshape(-1, 1))
        limpieza = np.clip(1.0 - (labels == -1).sum() / len(labels) / 0.40, 0.0, 1.0) * 20.0
    else: limpieza = 10.0
    clima = np.clip(1.0 - horas_calor / umbral_clima, 0.0, 1.0) * 10.0
    total = int(round(vigor + estabilidad + limpieza + clima))
    return {"total": total, "vigor": round(vigor, 1), "estabilidad": round(estabilidad, 1),
            "limpieza": round(limpieza, 1), "clima": round(clima, 1)}

# ============================================================================
# 6. MAPA INTERACTIVO CON AMBIENTES
# ============================================================================

def generar_mapa_offline(lote_gdf, cultivo="maiz", score=None, conf=None,
                          zonas_gdf=None, output_path="mapa.html"):
    import os
    centroide = lote_gdf.geometry.iloc[0].centroid
    m = folium.Map(location=[centroide.y, centroide.x], zoom_start=15, tiles='OpenStreetMap')
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
        attr='Google Satellite', name='Satelite (requiere senal)', overlay=False
    ).add_to(m)
    ZONA_COLORS = {'A': '#40916C', 'B': '#F4A261', 'C': '#D62828'}
    ZONA_LABELS = {'A': 'Zona A - Alto potencial', 'B': 'Zona B - Variable', 'C': 'Zona C - Limitante'}
    csv_criticos_path = None
    if zonas_gdf is not None:
        for zona in ['A', 'B']:
            color  = ZONA_COLORS[zona]
            subset = zonas_gdf[zonas_gdf['zona'] == zona]
            if subset.empty: continue
            fg = folium.FeatureGroup(name=ZONA_LABELS[zona], show=True)
            for _, row in subset.iterrows():
                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x], radius=5,
                    color=color, fill=True, fill_color=color, fill_opacity=0.7,
                    tooltip=("<b>Ambiente " + zona + "</b><br>NDVI: " + f"{row['ndvi']:.3f}" +
                             "<br>" + ZONA_LABELS[zona] +
                             "<br>Lat: " + f"{row.geometry.y:.5f}" + "<br>Lon: " + f"{row.geometry.x:.5f}"),
                ).add_to(fg)
            fg.add_to(m)
        zona_c = zonas_gdf[zonas_gdf['zona'] == 'C']
        if not zona_c.empty:
            fg_c = folium.FeatureGroup(name="Puntos criticos Zona C (" + str(len(zona_c)) + " pts)", show=True)
            for _, row in zona_c.iterrows():
                lat = row.geometry.y
                lon = row.geometry.x
                gmaps_url = "https://maps.google.com/?q=" + f"{lat:.6f}" + "," + f"{lon:.6f}"
                folium.CircleMarker(
                    location=[lat, lon], radius=7,
                    color='#D62828', fill=True, fill_color='#D62828', fill_opacity=0.9,
                    tooltip=("<b>Zona C - Limitante</b><br>NDVI: " + f"{row['ndvi']:.3f}" +
                             "<br>Lat: " + f"{lat:.5f}" + "<br>Lon: " + f"{lon:.5f}"),
                    popup=folium.Popup(
                        "<div style='font-family:sans-serif;font-size:13px'>"
                        "<b>Punto critico - Zona C</b><br>NDVI: <b>" + f"{row['ndvi']:.3f}" +
                        "</b><br>Lat: " + f"{lat:.6f}" + "<br>Lon: " + f"{lon:.6f}" + "<br><br>"
                        "<a href='" + gmaps_url + "' target='_blank' style='color:#D62828;font-weight:bold'>"
                        "Abrir en Google Maps</a></div>", max_width=220),
                ).add_to(fg_c)
            fg_c.add_to(m)
            base = os.path.splitext(output_path)[0]
            csv_criticos_path = base + "_ZonaC_criticos.csv"
            zona_c_export = zona_c.copy()
            zona_c_export['lat'] = zona_c_export.geometry.y
            zona_c_export['lon'] = zona_c_export.geometry.x
            zona_c_export[['lat', 'lon', 'ndvi', 'zona']].to_csv(csv_criticos_path, index=False)
            print("   Zona C: " + str(len(zona_c)) + " puntos criticos exportados a " + csv_criticos_path)
    score_str = str(score) if score else "N/D"
    folium.GeoJson(lote_gdf, name='Contorno del lote',
                   style_function=lambda x: {'fillColor': 'none', 'color': 'white', 'weight': 3, 'fillOpacity': 0},
                   tooltip=folium.Tooltip("<b>" + cultivo.capitalize() + "</b><br>Score AgroIA: " + score_str + "/100"),
                   ).add_to(m)
    plugins.LocateControl(auto_start=False, strings={"title": "Mi ubicacion en el lote"}).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.save(output_path)
    print("   Mapa guardado: " + output_path)
    return output_path, csv_criticos_path


# ============================================================================
# 7. PIPELINE PRINCIPAL
# ============================================================================

def run_agroia_pipeline(shp_path, cultivo="maiz", years=None):
    if years is None: years = list(range(2019, 2025))
    conf = CONFIG[cultivo]
    log  = []
    print("\n🔍 Validando shapefile...")
    lote, shp_warnings = validar_shapefile(shp_path)
    for w in shp_warnings:
        log.append(f"   ⚠ {w}")
    centroide = lote.geometry.iloc[0].centroid
    geom_ee   = ee.Geometry.Polygon(list(lote.geometry.iloc[0].exterior.coords))
    zona_utm   = int((centroide.x + 180) / 6) + 1
    hemisferio = 32700 if centroide.y < 0 else 32600
    epsg_utm   = hemisferio + zona_utm
    lote_utm   = lote.to_crs(f"EPSG:{epsg_utm}")
    hectareas  = round(lote_utm.geometry.area.sum() / 10_000, 2)
    print(f"✓ Lote: {hectareas} ha | EPSG:{epsg_utm} | Centroide: {centroide.y:.4f}, {centroide.x:.4f}")
    print(f"\n🌡️  Descargando NASA POWER ({len(years)} años)...")
    horas_calor_hist = {}
    for yr in years:
        h, status = get_nasa_climate_safe(centroide.y, centroide.x, yr, conf)
        horas_calor_hist[yr] = h
        msg = f"   {'✓' if status == 'ok' else '⚠'} NASA {yr}: {h:.1f}h{'' if status == 'ok' else f' ({status})'}"
        log.append(msg); print(msg)
    print(f"\n🛰️  Procesando GEE Sentinel-2 (mes critico: {conf['mes_critico']})...")
    ndvi_hist      = {}
    ndvi_mes_usado = {}
    for yr in years:
        val, status, msg = get_gee_ndvi_validado(geom_ee, yr, conf['mes_critico'], cultivo)
        if val is not None:
            ndvi_hist[yr]      = val
            ndvi_mes_usado[yr] = conf['mes_critico']
            log.append(f"   {msg}"); print(f"   {msg}")
        else:
            val_fb, mes_fb, aviso_fb = get_gee_ndvi_ventana(geom_ee, yr, conf['mes_critico'], max_delta=2)
            if val_fb is not None:
                status_fb, msg_fb = validar_ndvi(val_fb, cultivo, yr, mes_fb)
                if status_fb not in ('sospechoso_bajo', 'nulo'):
                    ndvi_hist[yr]      = val_fb
                    ndvi_mes_usado[yr] = mes_fb
                    log.append(f"   {aviso_fb}"); print(f"   {aviso_fb}")
                else:
                    log.append(f"   {msg_fb}"); print(f"   {msg_fb}")
            else:
                log.append(f"   {aviso_fb}"); print(f"   {aviso_fb}")
    año_actual = max(years)
    if año_actual not in ndvi_hist:
        raise ValueError(
            f"\n❌ Sin NDVI valido para {año_actual} ni en ventana ±2 meses.\n"
            f"   Causas: nubosidad extrema, lote sin cultivo, o shapefile incorrecto.")
    print("\n📊 Analizando ambientes espaciales...")
    zonas_gdf = None
    cv, _, _ = calcular_cv_gee(geom_ee, año_actual, conf['mes_critico'])
    es_variable = cv is not None and cv > 0.05
    if es_variable:
        zonas_gdf = zonificar_lote_gee(geom_ee, año_actual, conf['mes_critico'])
        print(f"   ✓ CV={cv:.3f} → Zonificación A/B/C activada")
    else:
        print(f"   ✓ CV={cv:.3f if cv else 'N/D'} → Lote homogéneo")
    score_dict = calcular_score(
        ndvi_hist[año_actual],
        horas_calor_hist[año_actual],
        list(ndvi_hist.values()),
        conf.get('umbral_clima', 40)
    )
    anos_excluidos = [y for y in years if y not in ndvi_hist]
    if anos_excluidos:
        print(f"   ⚠ Años excluidos del Score: {anos_excluidos}")
    print(f"\n{'='*50}")
    print(f"  ✅  SCORE AGROIA: {score_dict['total']}/100")
    print(f"      Vigor        {score_dict['vigor']:>5.1f} / 40")
    print(f"      Estabilidad  {score_dict['estabilidad']:>5.1f} / 30")
    print(f"      Limpieza IA  {score_dict['limpieza']:>5.1f} / 20")
    print(f"      Clima        {score_dict['clima']:>5.1f} / 10")
    print(f"{'='*50}\n")
    return {
        "cultivo":             cultivo,
        "hectareas":           hectareas,
        "epsg_utm":            epsg_utm,
        "centroide":           (centroide.y, centroide.x),
        "score":               score_dict,
        "ndvi_historico":      ndvi_hist,
        "ndvi_hist_bruto":     ndvi_hist,
        "anos_excluidos":      anos_excluidos,
        "ndvi_critico_actual": ndvi_hist[año_actual],
        "horas_calor_hist":    horas_calor_hist,
        "horas_calor_actual":  horas_calor_hist[año_actual],
        "cv":                  cv,
        "es_variable":         es_variable,
        "lote_gdf":            lote,
        "zonas_gdf":           zonas_gdf,
        "geom_ee":             geom_ee,
        "conf":                conf,
        "log":                 log,
    }

def verificar_resultado(res):
    print(f"✓ Score: {res['score']['total']} | CV: {res['cv']:.3f} | {'Variable' if res['es_variable'] else 'Homogéneo'}")

# -*- coding: utf-8 -*-
"""
AGROIA - PDF ASSEMBLER V2.5
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from io import BytesIO
from datetime import datetime

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib.colors import HexColor, white, black
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image,
    Table, TableStyle, HRFlowable, KeepTogether, PageBreak
)
from reportlab.pdfgen import canvas as rl_canvas

# ============================================================================
# PALETA Y CONSTANTES
# ============================================================================

C_PRIMARY    = HexColor("#1B4332")
C_SECONDARY  = HexColor("#40916C")
C_ACCENT_BG  = HexColor("#D8F3DC")
C_ALERT      = HexColor("#D62828")
C_WARN       = HexColor("#F4A261")
C_GOLD       = HexColor("#E9C46A")
C_NEUTRAL    = HexColor("#F8F9FA")
C_TEXT       = HexColor("#212529")
C_MUTED      = HexColor("#6C757D")
C_BORDER     = HexColor("#DEE2E6")

ZONA_COLORS  = {"A": "#40916C", "B": "#F4A261", "C": "#D62828"}
ZONA_LABELS  = {"A": "Zona A — Alto potencial", "B": "Zona B — Variable", "C": "Zona C — Limitante"}

VERSION      = "2.5.0"
PAGE_W, PAGE_H = A4

# ============================================================================
# ESTILOS
# ============================================================================

def build_styles():
    return {
        "cover_title": ParagraphStyle("CoverTitle", fontSize=32, fontName="Helvetica-Bold", textColor=white, leading=36, spaceAfter=6),
        "cover_sub":   ParagraphStyle("CoverSub",   fontSize=13, fontName="Helvetica",      textColor=HexColor("#D8F3DC"), leading=18, spaceAfter=4),
        "cover_meta":  ParagraphStyle("CoverMeta",  fontSize=9,  fontName="Helvetica",      textColor=HexColor("#95D5B2"), leading=13),
        "h2": ParagraphStyle("H2", fontSize=11, fontName="Helvetica-Bold", textColor=white, spaceAfter=10, spaceBefore=4, backColor=C_PRIMARY, leftIndent=-16, rightIndent=-16, borderPad=7),
        "h3": ParagraphStyle("H3", fontSize=10, fontName="Helvetica-Bold", textColor=C_PRIMARY, spaceAfter=4, spaceBefore=10),
        "body": ParagraphStyle("Body", fontSize=9, fontName="Helvetica", textColor=C_TEXT, leading=14, spaceAfter=5, alignment=TA_JUSTIFY),
        "body_bold": ParagraphStyle("BodyBold", fontSize=9, fontName="Helvetica-Bold", textColor=C_TEXT, leading=14, spaceAfter=4),
        "alert_red":  ParagraphStyle("AlertRed",  fontSize=9, fontName="Helvetica-Bold", textColor=C_ALERT, spaceAfter=4, leftIndent=8),
        "alert_warn": ParagraphStyle("AlertWarn", fontSize=9, fontName="Helvetica-Bold", textColor=HexColor("#8B4513"), spaceAfter=4, leftIndent=8),
        "alert_ok":   ParagraphStyle("AlertOk",   fontSize=9, fontName="Helvetica",      textColor=HexColor("#1B4332"), spaceAfter=4, leftIndent=8),
        "caption": ParagraphStyle("Caption", fontSize=7.5, fontName="Helvetica-Oblique", textColor=C_MUTED, spaceAfter=8, alignment=TA_CENTER),
        "meta":    ParagraphStyle("Meta",    fontSize=7,   fontName="Helvetica",          textColor=C_MUTED, alignment=TA_CENTER),
        "table_header": ParagraphStyle("TH", fontSize=8.5, fontName="Helvetica-Bold", textColor=white),
        "table_cell":   ParagraphStyle("TC", fontSize=8.5, fontName="Helvetica",      textColor=C_TEXT),
        "score_big":    ParagraphStyle("ScoreBig",   fontSize=48, fontName="Helvetica-Bold", textColor=white, alignment=TA_CENTER, leading=52),
        "score_label":  ParagraphStyle("ScoreLabel", fontSize=10, fontName="Helvetica",      textColor=HexColor("#D8F3DC"), alignment=TA_CENTER),
        "log": ParagraphStyle("Log", fontSize=7, fontName="Courier", textColor=C_MUTED, leading=10, spaceAfter=2),
    }


def fig_to_rl_image(fig, width_cm=16, dpi=150):
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight", facecolor="white", edgecolor="none")
    buf.seek(0)
    plt.close(fig)
    w = width_cm * cm
    h = w * (fig.get_figheight() / fig.get_figwidth())
    return Image(buf, width=w, height=h)


# ============================================================================
# GRÁFICOS
# ============================================================================

def plot_score_gauge(score_dict):
    total = score_dict["total"]
    color_score = "#40916C" if total >= 70 else "#F4A261" if total >= 45 else "#D62828"
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={'width_ratios': [1, 1.4]})
    fig.patch.set_facecolor('white')
    ax = axes[0]
    ax.set_aspect('equal')
    ax.axis('off')
    theta_start = np.pi
    theta_end   = 0
    theta_score = theta_start - (total / 100) * np.pi
    theta_bg = np.linspace(theta_start, theta_end, 200)
    r_out, r_in = 1.0, 0.62
    ax.fill_between(np.concatenate([r_out * np.cos(theta_bg), r_in * np.cos(theta_bg[::-1])]),
                    np.concatenate([r_out * np.sin(theta_bg), r_in * np.sin(theta_bg[::-1])]),
                    color="#E9ECEF", zorder=1)
    theta_fill = np.linspace(theta_start, theta_score, 200)
    ax.fill_between(np.concatenate([r_out * np.cos(theta_fill), r_in * np.cos(theta_fill[::-1])]),
                    np.concatenate([r_out * np.sin(theta_fill), r_in * np.sin(theta_fill[::-1])]),
                    color=color_score, zorder=2)
    ax.text(0, 0.18, str(total), ha='center', va='center', fontsize=42, fontweight='bold', color=color_score)
    ax.text(0, -0.08, "/ 100", ha='center', va='center', fontsize=13, color='#6C757D')
    ax.text(0, -0.30, "Score AgroIA", ha='center', va='center', fontsize=10, color='#495057')
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-0.5, 1.2)
    ax2 = axes[1]
    ax2.set_facecolor('white')
    componentes = [("Vigor", score_dict["vigor"], 40, "#40916C"),
                   ("Estabilidad", score_dict["estabilidad"], 30, "#2D6A4F"),
                   ("Limpieza IA", score_dict["limpieza"], 20, "#74C69D"),
                   ("Clima", score_dict["clima"], 10, "#F4A261")]
    for i, (label, val, max_val, color) in enumerate(componentes):
        ax2.barh(i, max_val, color="#E9ECEF", height=0.55, zorder=1)
        ax2.barh(i, val, color=color, height=0.55, zorder=2)
        ax2.text(max_val + 0.5, i, f"{val:.1f}/{max_val}", va='center', fontsize=9, color='#495057')
        ax2.text(-0.5, i, label, va='center', ha='right', fontsize=9, color='#212529', fontweight='bold')
    ax2.set_xlim(-8, 48)
    ax2.set_ylim(-0.6, len(componentes) - 0.4)
    ax2.axis('off')
    plt.tight_layout(pad=1.0)
    return fig


def plot_ndvi_historico(ndvi_hist, ndvi_hist_bruto, anos_excluidos, cultivo, conf, año_actual):
    fig, ax = plt.subplots(figsize=(12, 4))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('#FAFAFA')
    years_all    = sorted(ndvi_hist_bruto.keys())
    ndvi_valid   = [ndvi_hist.get(y) for y in years_all]
    vals_validos = [v for v in ndvi_valid if v is not None]
    media = np.mean(vals_validos) if vals_validos else None
    xs_v = [y for y, v in zip(years_all, ndvi_valid) if v is not None]
    ys_v = [v for v in ndvi_valid if v is not None]
    ax.plot(xs_v, ys_v, 'o-', color='#40916C', linewidth=2, markersize=7, zorder=3, label='NDVI válido (en Score)')
    xs_e = [y for y in anos_excluidos if ndvi_hist_bruto.get(y) is not None]
    ys_e = [ndvi_hist_bruto[y] for y in xs_e]
    if xs_e:
        ax.scatter(xs_e, ys_e, color='#ADB5BD', marker='X', s=80, zorder=4, label='Excluido (no confiable)')
    if len(vals_validos) >= 3:
        sigma = np.std(vals_validos)
        ax.axhspan(media - sigma, media + sigma, color='#D8F3DC', alpha=0.4, label='±1σ histórico')
    if media:
        ax.axhline(media, color='#2D6A4F', linestyle='--', linewidth=1.2, alpha=0.7, label=f'Media: {media:.3f}')
    if año_actual in ndvi_hist:
        ax.scatter([año_actual], [ndvi_hist[año_actual]], color='#D62828', s=120, zorder=5, label=f'Año actual ({año_actual})')
    mes_nombre = {1:"Enero", 2:"Feb", 3:"Mar", 10:"Oct", 11:"Nov", 12:"Dic"}
    mes_str = mes_nombre.get(conf['mes_critico'], f"Mes {conf['mes_critico']}")
    ax.set_title(f"NDVI en ventana crítica ({mes_str}) — {cultivo.capitalize()} | Historial {min(years_all)}–{max(years_all)}", fontsize=10, loc='left', pad=8)
    ax.set_ylabel("NDVI medio del lote", fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=8, loc='lower right', framealpha=0.8)
    ax.grid(axis='y', linestyle=':', alpha=0.4)
    plt.tight_layout()
    return fig


def plot_estres_termico(horas_calor_hist, conf, año_actual):
    years = sorted(horas_calor_hist.keys())
    horas = [horas_calor_hist[y] for y in years]
    media = np.mean([h for h in horas if h > 0]) if any(h > 0 for h in horas) else 0
    colores = ['#D62828' if y == año_actual else '#F4A261' if h > media * 1.2 else '#74C69D' for y, h in zip(years, horas)]
    fig, ax = plt.subplots(figsize=(11, 3.8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('#FAFAFA')
    bars = ax.bar(years, horas, color=colores, edgecolor='white', linewidth=0.5, width=0.65)
    for bar, h in zip(bars, horas):
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, f"{h:.1f}h", ha='center', va='bottom', fontsize=8, color='#495057')
    if media > 0:
        ax.axhline(media, color='#2D6A4F', linestyle='--', linewidth=1.2, label=f'Media: {media:.1f}h')
    umbral = conf['umbral_calor']
    ax.set_title(f"Horas estimadas sobre {umbral}°C en ventana crítica (sinusoidal NASA POWER)", fontsize=10, loc='left', pad=8)
    ax.set_ylabel("Horas sobre umbral", fontsize=9)
    ax.tick_params(labelsize=8)
    if media > 0:
        ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle=':', alpha=0.4)
    plt.tight_layout()
    return fig


# ============================================================================
# TEXTO DINÁMICO
# ============================================================================

def generar_texto_diagnostico(res):
    score   = res["score"]["total"]
    cultivo = res["cultivo"].capitalize()
    ha      = res["hectareas"]
    ndvi    = res["ndvi_critico_actual"]
    horas   = res["horas_calor_actual"]
    cv      = res["cv"]
    año     = max(res["ndvi_historico"].keys())
    nivel   = "alto" if score >= 70 else "intermedio" if score >= 45 else "bajo"
    zona_str = (
        "La variabilidad espacial interna (CV={:.2f}) es suficiente para activar la <b>zonificación A/B/C</b> del lote.".format(cv)
        if res["es_variable"] else
        "El lote es <b>espacialmente homogéneo</b> (CV={:.3f}), toda la superficie responde de manera uniforme.".format(cv)
    )
    return (f"El lote de <b>{ha} ha</b> de <b>{cultivo}</b> obtuvo un <b>Score AgroIA de {score}/100</b> "
            f"para la campaña <b>{año}</b>, calificando como potencial <b>{nivel}</b>. "
            f"El NDVI mediano en la ventana crítica fue de <b>{ndvi:.3f}</b>. "
            f"La estimación sinusoidal sobre datos NASA POWER detectó "
            f"<b>{horas:.1f} horas por encima de {res['conf']['umbral_calor']}°C</b> "
            f"en el mes de mayor riesgo térmico. {zona_str}")


def generar_recomendaciones(res):
    styles = build_styles()
    recs   = []
    score  = res["score"]["total"]
    horas  = res["horas_calor_actual"]
    conf   = res["conf"]
    umbral = conf["umbral_calor"]
    ndvi   = res["ndvi_critico_actual"]
    if score >= 70:
        recs.append((styles["alert_ok"], "✓ <b>Zona de Alto Potencial:</b> mantener la inversión en insumos al nivel máximo. El historial de NDVI valida la respuesta consistente del suelo."))
    elif score >= 45:
        recs.append((styles["alert_warn"], "▲ <b>Potencial Intermedio:</b> evaluar ajuste de densidad de siembra y fertilización nitrogenada según disponibilidad hídrica esperada."))
    else:
        recs.append((styles["alert_red"], "✗ <b>Zona Limitante:</b> planteo defensivo recomendado. No sobre-invertir en insumos variables. Priorizar cobertura de riesgo (seguro agrícola)."))
    NDVI_ALERTA = {"maiz": 0.50, "soja": 0.45, "trigo": 0.40}
    umbral_ndvi = NDVI_ALERTA.get(res["cultivo"], 0.50)
    if ndvi < umbral_ndvi:
        ndvi_hist_vals = list(res["ndvi_historico"].values())
        ndvi_max_hist  = max(ndvi_hist_vals) if ndvi_hist_vals else ndvi
        caida_pct      = (1 - ndvi / ndvi_max_hist) * 100 if ndvi_max_hist > 0 else 0
        recs.append((styles["alert_warn"],
            f"▲ <b>NDVI bajo en ventana crítica:</b> el valor registrado ({ndvi:.3f}) está por debajo del umbral "
            f"({umbral_ndvi}) para {res['cultivo']}. Caída del {caida_pct:.0f}% respecto al mejor año histórico "
            f"({ndvi_max_hist:.3f}). <b>Se recomienda recorrida de campo.</b>"))
    if horas > 10:
        recs.append((styles["alert_red"], f"✗ <b>Alerta de Sopladura:</b> {horas:.1f} horas estimadas sobre {umbral}°C en ventana crítica. Riesgo de aborto de granos documentado por literatura INTA/Nebraska ({conf['biblio']}). Revisar campaña con seguro."))
    elif horas > 5:
        recs.append((styles["alert_warn"], f"▲ <b>Estrés Térmico Moderado:</b> {horas:.1f} horas sobre {umbral}°C. Monitorear llenado de granos."))
    else:
        recs.append((styles["alert_ok"], f"✓ <b>Sin estrés térmico significativo</b> en ventana crítica ({horas:.1f}h sobre {umbral}°C)."))
    if res["anos_excluidos"]:
        recs.append((styles["alert_warn"], f"▲ <b>Datos excluidos:</b> los años {res['anos_excluidos']} fueron descartados del Score por NDVI fuera de rango."))
    return recs


# ============================================================================
# HEADER / FOOTER
# ============================================================================

class PageDecorator:
    def __init__(self, cultivo, lote_id, score, fecha):
        self.cultivo  = cultivo.capitalize()
        self.lote_id  = lote_id
        self.score    = score
        self.fecha    = fecha
        self._is_first = True

    def __call__(self, canv, doc):
        if self._is_first:
            self._is_first = False
            return
        canv.saveState()
        w = PAGE_W
        canv.setFillColor(C_PRIMARY)
        canv.rect(1.8*cm, PAGE_H - 1.5*cm, w - 3.6*cm, 0.4*cm, fill=1, stroke=0)
        canv.setFont("Helvetica-Bold", 8)
        canv.setFillColor(white)
        canv.drawString(2.2*cm, PAGE_H - 1.28*cm, f"AgroIA Report — {self.cultivo} | {self.lote_id}")
        canv.setFont("Helvetica", 8)
        canv.drawRightString(w - 2.2*cm, PAGE_H - 1.28*cm, f"Score: {self.score}/100")
        canv.setFillColor(C_MUTED)
        canv.setFont("Helvetica", 7)
        canv.drawString(1.8*cm, 1.1*cm, f"Generado: {self.fecha} | v{VERSION} | NASA POWER + GEE Sentinel-2")
        canv.drawRightString(w - 1.8*cm, 1.1*cm, f"Pág. {doc.page}")
        canv.setStrokeColor(C_BORDER)
        canv.setLineWidth(0.3)
        canv.line(1.8*cm, 1.5*cm, w - 1.8*cm, 1.5*cm)
        canv.restoreState()


# ============================================================================
# PORTADA
# ============================================================================

def build_cover_page(canv, res, fecha_str, lote_id):
    w, h = PAGE_W, PAGE_H
    score = res["score"]["total"]
    color_score = HexColor("#40916C") if score >= 70 else HexColor("#F4A261") if score >= 45 else HexColor("#D62828")
    canv.setFillColor(C_PRIMARY)
    canv.rect(0, 0, w, h, fill=1, stroke=0)
    canv.setFillColor(C_SECONDARY)
    canv.rect(0, 0, w, 2.8*cm, fill=1, stroke=0)
    cx, cy, r = w - 4.5*cm, h - 5.5*cm, 2.6*cm
    canv.setFillColor(color_score)
    canv.circle(cx, cy, r, fill=1, stroke=0)
    canv.setFont("Helvetica-Bold", 36)
    canv.setFillColor(white)
    canv.drawCentredString(cx, cy + 0.3*cm, str(score))
    canv.setFont("Helvetica", 11)
    canv.drawCentredString(cx, cy - 0.7*cm, "/ 100")
    canv.setFont("Helvetica", 9)
    canv.setFillColor(HexColor("#D8F3DC"))
    canv.drawCentredString(cx, cy - 1.4*cm, "Score AgroIA")
    canv.setFont("Helvetica-Bold", 30)
    canv.setFillColor(white)
    canv.drawString(2*cm, h - 3.5*cm, "AgroIA Report")
    canv.setFont("Helvetica", 16)
    canv.setFillColor(HexColor("#95D5B2"))
    canv.drawString(2*cm, h - 4.3*cm, f"Gestión de Riesgo Agronómico — {res['cultivo'].capitalize()}")
    canv.setFont("Helvetica-Bold", 11)
    canv.setFillColor(HexColor("#D8F3DC"))
    canv.drawString(2*cm, h - 5.8*cm, "Datos del lote")
    canv.setLineWidth(0.5)
    canv.setStrokeColor(C_SECONDARY)
    canv.line(2*cm, h - 5.95*cm, 10*cm, h - 5.95*cm)
    detalles = [
        ("ID del lote",       lote_id),
        ("Superficie",        f"{res['hectareas']} ha"),
        ("Cultivo",           res['cultivo'].capitalize()),
        ("Coordenadas",       f"{res['centroide'][0]:.4f}°, {res['centroide'][1]:.4f}°"),
        ("Proyección UTM",    f"EPSG:{res['epsg_utm']}"),
        ("Período analizado", f"{min(res['ndvi_historico'].keys())}–{max(res['ndvi_historico'].keys())}"),
    ]
    y = h - 6.5*cm
    for label, valor in detalles:
        canv.setFont("Helvetica", 9)
        canv.setFillColor(HexColor("#95D5B2"))
        canv.drawString(2*cm, y, f"{label}:")
        canv.setFont("Helvetica-Bold", 9)
        canv.setFillColor(white)
        canv.drawString(6.5*cm, y, valor)
        y -= 0.6*cm
    canv.setFont("Helvetica-Bold", 10)
    canv.setFillColor(HexColor("#D8F3DC"))
    canv.drawString(2*cm, h - 10.4*cm, "Composición del Score")
    canv.line(2*cm, h - 10.6*cm, 10*cm, h - 10.6*cm)
    comp = [("40% Vigor", f"{res['score']['vigor']:.1f} pts"),
            ("30% Estabilidad", f"{res['score']['estabilidad']:.1f} pts"),
            ("20% Limpieza IA", f"{res['score']['limpieza']:.1f} pts"),
            ("10% Clima", f"{res['score']['clima']:.1f} pts")]
    y = h - 11.2*cm
    for label, valor in comp:
        canv.setFont("Helvetica", 8.5)
        canv.setFillColor(HexColor("#95D5B2"))
        canv.drawString(2*cm, y, label)
        canv.setFont("Helvetica-Bold", 8.5)
        canv.setFillColor(white)
        canv.drawRightString(9.5*cm, y, valor)
        y -= 0.5*cm
    canv.setFont("Helvetica", 8)
    canv.setFillColor(white)
    canv.drawCentredString(w/2, 2.0*cm, f"Generado: {fecha_str}  |  v{VERSION}  |  NASA POWER · GEE Sentinel-2 · IA (IsolationForest)")
    canv.setFont("Helvetica", 7.5)
    canv.setFillColor(HexColor("#95D5B2"))
    canv.drawCentredString(w/2, 1.3*cm, "Estimaciones basadas en interpolación sinusoidal Tmax/Tmin. Citar: INTA Marcos Juárez / Univ. Nebraska.")


# ============================================================================
# ASSEMBLER PRINCIPAL
# ============================================================================

def build_report(res, output_path=None, lote_id=None):
    styles    = build_styles()
    cultivo   = res["cultivo"]
    año_act   = max(res["ndvi_historico"].keys())
    fecha_str = datetime.now().strftime("%d/%m/%Y %H:%M")
    umbral_clima = res['conf'].get('umbral_clima', 40)   # ← por cultivo

    if lote_id is None:
        lote_id = f"Lote {cultivo.capitalize()} {año_act}"
    if output_path is None:
        output_path = f"AgroIA_{cultivo}_{año_act}.pdf"

    from reportlab.pdfgen import canvas as rl_canvas_mod
    from pypdf import PdfWriter, PdfReader
    import tempfile, os

    cover_path = tempfile.mktemp(suffix=".pdf")
    body_path  = tempfile.mktemp(suffix=".pdf")

    c = rl_canvas_mod.Canvas(cover_path, pagesize=A4)
    build_cover_page(c, res, fecha_str, lote_id)
    c.showPage()
    c.save()

    decorator = PageDecorator(cultivo, lote_id, res["score"]["total"], fecha_str)
    doc = SimpleDocTemplate(body_path, pagesize=A4,
                            leftMargin=1.8*cm, rightMargin=1.8*cm,
                            topMargin=2.2*cm, bottomMargin=2.0*cm)
    story = []

    # Sección 1: Diagnóstico
    story.append(Paragraph("1. Diagnóstico integrado del lote", styles["h2"]))
    story.append(Spacer(1, 0.2*cm))
    story.append(Paragraph(generar_texto_diagnostico(res), styles["body"]))
    story.append(Spacer(1, 0.4*cm))
    story.append(Paragraph("Composición del Score AgroIA", styles["h3"]))
    story.append(fig_to_rl_image(plot_score_gauge(res["score"]), width_cm=15))
    story.append(Paragraph("Fig. 1 — Score AgroIA (0–100) y desglose: Vigor (40%), Estabilidad (30%), Limpieza IA (20%), Clima (10%).", styles["caption"]))
    story.append(Spacer(1, 0.5*cm))

    # Sección 2: NDVI
    story.append(Paragraph("2. Análisis satelital — NDVI histórico (GEE Sentinel-2)", styles["h2"]))
    story.append(Spacer(1, 0.2*cm))
    años_validos = len(res["ndvi_historico"])
    años_excl    = len(res["anos_excluidos"])
    story.append(Paragraph(
        f"Serie construida con <b>{años_validos} años válidos</b> de mosaicos medianos Sentinel-2 SR. "
        f"{f'<b>{años_excl} años excluidos</b> por NDVI fuera de rango agronómico.' if años_excl else ''}",
        styles["body"]))
    story.append(Spacer(1, 0.3*cm))
    story.append(fig_to_rl_image(plot_ndvi_historico(res["ndvi_historico"], res["ndvi_hist_bruto"], res["anos_excluidos"], cultivo, res["conf"], año_act), width_cm=16))
    story.append(Paragraph(f"Fig. 2 — NDVI mediano en ventana crítica (mes {res['conf']['mes_critico']}) por año.", styles["caption"]))
    story.append(Spacer(1, 0.5*cm))

    # Tabla NDVI simple
    story.append(Paragraph("Detalle por campaña", styles["h3"]))
    tabla_data = [[Paragraph(h, styles["table_header"]) for h in ["Año", "NDVI crítico", "Estado", "En Score"]]]
    for yr in sorted(res["ndvi_hist_bruto"].keys()):
        val      = res["ndvi_hist_bruto"].get(yr)
        en_score = "Sí" if yr in res["ndvi_historico"] else "No"
        estado   = "Válido" if yr in res["ndvi_historico"] else "Excluido" if yr in res["anos_excluidos"] else "Sin dato"
        tabla_data.append([Paragraph(str(yr), styles["table_cell"]),
                           Paragraph(f"{val:.3f}" if val else "—", styles["table_cell"]),
                           Paragraph(estado, styles["table_cell"]),
                           Paragraph(en_score, styles["table_cell"])])
    tabla = Table(tabla_data, colWidths=[3*cm, 4*cm, 4*cm, 3.5*cm])
    tabla.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,0), C_PRIMARY), ("TEXTCOLOR", (0,0), (-1,0), white),
                                ("ROWBACKGROUNDS", (0,1), (-1,-1), [white, C_ACCENT_BG]),
                                ("GRID", (0,0), (-1,-1), 0.3, C_BORDER),
                                ("ALIGN", (0,0), (-1,-1), "CENTER"), ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
                                ("TOPPADDING", (0,0), (-1,-1), 5), ("BOTTOMPADDING", (0,0), (-1,-1), 5)]))
    story.append(tabla)
    story.append(Spacer(1, 0.6*cm))

    # Tabla histórica de scores (usa umbral_clima del cultivo)
    story.append(Spacer(1, 0.5*cm))
    story.append(Paragraph("Historial de Scores por Campaña", styles["h3"]))
    story.append(Spacer(1, 0.2*cm))
    anos_hist = sorted(res["ndvi_historico"].keys())
    filas = [[Paragraph(h, styles["table_header"]) for h in
              ["Año", "NDVI", "Estrés (h)", "Vigor", "Estab.", "Limp.", "Clima", "Score"]]]
    for a in anos_hist:
        ndvi  = res["ndvi_historico"][a]
        horas = res["horas_calor_hist"].get(a, 0)
        previos = [res["ndvi_historico"][y] for y in anos_hist if y < a and y not in res.get("anos_excluidos", [])]
        vigor = float(np.clip(ndvi / 0.9, 0, 1) * 40)
        clima = float(np.clip(1 - (horas or 0) / umbral_clima, 0, 1) * 10)   # ← umbral por cultivo
        if len(previos) >= 2:
            cv_p = np.std(previos) / np.mean(previos) if np.mean(previos) > 0 else 0
            est  = float(np.clip(1 - cv_p / 0.45, 0, 1) * 30)
        else:
            est = float(np.clip(15.0 + (ndvi - 0.6) * 10, 10, 25))
        if len(previos) >= 4:
            iso    = IsolationForest(contamination=0.2, random_state=42)
            labels = iso.fit_predict(np.array(previos).reshape(-1, 1))
            limp   = float(np.clip(1 - (labels==-1).sum()/len(labels)/0.40, 0, 1) * 20)
        else:
            limp = float(np.clip(10.0 + (ndvi - 0.6) * 5, 8, 15))
        total = int(round(vigor + est + limp + clima))
        filas.append([Paragraph(str(a), styles["table_cell"]),
                      Paragraph(f"{ndvi:.3f}", styles["table_cell"]),
                      Paragraph(f"{horas:.1f}", styles["table_cell"]),
                      Paragraph(f"{vigor:.1f}", styles["table_cell"]),
                      Paragraph(f"{est:.1f}", styles["table_cell"]),
                      Paragraph(f"{limp:.1f}", styles["table_cell"]),
                      Paragraph(f"{clima:.1f}", styles["table_cell"]),
                      Paragraph(f"{total}/100", styles["table_cell"])])
    tabla_hist = Table(filas, colWidths=[1.5*cm, 1.6*cm, 1.9*cm, 1.5*cm, 1.5*cm, 1.5*cm, 1.5*cm, 1.5*cm])
    tabla_hist.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,0), C_PRIMARY), ("TEXTCOLOR", (0,0), (-1,0), white),
                                     ("ROWBACKGROUNDS", (0,1), (-1,-1), [white, C_ACCENT_BG]),
                                     ("GRID", (0,0), (-1,-1), 0.3, C_BORDER),
                                     ("ALIGN", (0,0), (-1,-1), "CENTER"), ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
                                     ("FONTSIZE", (0,0), (-1,-1), 7.5),
                                     ("TOPPADDING", (0,0), (-1,-1), 4), ("BOTTOMPADDING", (0,0), (-1,-1), 4)]))
    story.append(tabla_hist)
    story.append(Spacer(1, 0.3*cm))

    # Sección 3: Clima
    story.append(Paragraph("3. Análisis climático — Estrés térmico NASA POWER", styles["h2"]))
    story.append(Spacer(1, 0.2*cm))
    story.append(Paragraph(
        f"Fuente: NASA POWER (~0.5°). Método: interpolación sinusoidal T(h) = Tmean + Tamp·cos(π·(h−14)/12). "
        f"Umbral térmico: <b>{res['conf']['umbral_calor']}°C</b>. "
        f"Umbral de penalización Score Clima: <b>{umbral_clima}h</b> "
        f"(ref. INTA Marcos Juárez / Nebraska: {res['conf']['biblio']}). "
        f"<b>Nota:</b> estimaciones basadas en Tmax/Tmin diarios — no reemplazan estación local.",
        styles["body"]))
    story.append(Spacer(1, 0.3*cm))
    story.append(fig_to_rl_image(plot_estres_termico(res["horas_calor_hist"], res["conf"], año_act), width_cm=15))
    story.append(Paragraph(
        f"Fig. 3 — Horas sobre {res['conf']['umbral_calor']}°C en mes crítico (mes {res['conf']['mes_critico']}). "
        f"Score Clima = 0 cuando horas ≥ {umbral_clima}h.", styles["caption"]))
    story.append(Spacer(1, 0.6*cm))

    # Sección 4: Recomendaciones
    story.append(Paragraph("4. Recomendaciones de manejo", styles["h2"]))
    story.append(Spacer(1, 0.2*cm))
    for estilo, texto in generar_recomendaciones(res):
        story.append(Paragraph(texto, estilo))
        story.append(Spacer(1, 0.15*cm))
    if res["es_variable"]:
        story.append(Spacer(1, 0.3*cm))
        story.append(Paragraph("Zonificación A/B/C activada", styles["h3"]))
        story.append(Paragraph("El CV espacial supera el umbral de homogeneidad. La zonificación se visualiza en el mapa HTML adjunto.", styles["body"]))
        zona_tabla = [
            [Paragraph(h, styles["table_header"]) for h in ["Zona", "Descripción", "Recomendación de manejo"]],
            [Paragraph("A", styles["table_cell"]), Paragraph("Alto potencial — NDVI estable y elevado", styles["table_cell"]), Paragraph("Invertir al máximo. Máxima densidad y dosis.", styles["table_cell"])],
            [Paragraph("B", styles["table_cell"]), Paragraph("Potencial variable — responde a buena campaña", styles["table_cell"]), Paragraph("Planteo estándar. Monitorear en año seco.", styles["table_cell"])],
            [Paragraph("C", styles["table_cell"]), Paragraph("Zona limitante — NDVI bajo o inestable", styles["table_cell"]), Paragraph("Planteo defensivo. No sobre-invertir.", styles["table_cell"])],
        ]
        zt = Table(zona_tabla, colWidths=[2*cm, 6*cm, 6.5*cm])
        zt.setStyle(TableStyle([
            ("BACKGROUND", (0,0), (-1,0), C_PRIMARY),
            ("BACKGROUND", (0,1), (0,1), HexColor(ZONA_COLORS["A"])),
            ("BACKGROUND", (0,2), (0,2), HexColor(ZONA_COLORS["B"])),
            ("BACKGROUND", (0,3), (0,3), HexColor(ZONA_COLORS["C"])),
            ("TEXTCOLOR", (0,1), (0,-1), white),
            ("ROWBACKGROUNDS", (1,1), (-1,-1), [white, C_NEUTRAL, white]),
            ("GRID", (0,0), (-1,-1), 0.3, C_BORDER),
            ("ALIGN", (0,0), (-1,-1), "CENTER"), ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
            ("TOPPADDING", (0,0), (-1,-1), 6), ("BOTTOMPADDING", (0,0), (-1,-1), 6)]))
        story.append(zt)
        story.append(Spacer(1, 0.5*cm))
    else:
        story.append(Spacer(1, 0.2*cm))
        story.append(Paragraph(f"✓ <b>Lote Homogéneo (CV = {res['cv']:.3f}):</b> manejo uniforme apropiado.", styles["alert_ok"]))

    # Sección 5: Metodología
    story.append(PageBreak())
    story.append(Paragraph("5. Cuadro metodológico y trazabilidad", styles["h2"]))
    story.append(Spacer(1, 0.2*cm))
    story.append(Paragraph("Cada valor tiene trazabilidad directa a la fuente y al método de cálculo.", styles["body"]))
    story.append(Spacer(1, 0.3*cm))
    met_tabla = [
        [Paragraph(h, styles["table_header"]) for h in ["Módulo", "Fuente", "Método", "Limitación conocida"]],
        [Paragraph("NDVI histórico", styles["table_cell"]), Paragraph("GEE Sentinel-2 SR", styles["table_cell"]), Paragraph("Mosaico mediano, cloud mask <20%", styles["table_cell"]), Paragraph("Sin imágenes en períodos >80% nubosidad", styles["table_cell"])],
        [Paragraph("Estrés térmico", styles["table_cell"]), Paragraph("NASA POWER (~0.5°)", styles["table_cell"]), Paragraph("Sinusoidal Tmax/Tmin → horas >umbral", styles["table_cell"]), Paragraph("Estimación; subestima picos convectivos", styles["table_cell"])],
        [Paragraph("Score Vigor (40%)", styles["table_cell"]), Paragraph("GEE NDVI", styles["table_cell"]), Paragraph("NDVI_crítico / 0.9 × 40", styles["table_cell"]), Paragraph("Normalización sobre máximo teórico 0.9", styles["table_cell"])],
        [Paragraph("Score Estabilidad (30%)", styles["table_cell"]), Paragraph("Serie NDVI histórica", styles["table_cell"]), Paragraph("(1 − CV/0.45) × 30", styles["table_cell"]), Paragraph("Requiere mínimo 3 años de datos", styles["table_cell"])],
        [Paragraph("Score Limpieza IA (20%)", styles["table_cell"]), Paragraph("Serie NDVI histórica", styles["table_cell"]), Paragraph("IsolationForest (contam.=0.2)", styles["table_cell"]), Paragraph("Requiere mínimo 4 años", styles["table_cell"])],
        [Paragraph("Score Clima (10%)", styles["table_cell"]), Paragraph("NASA POWER", styles["table_cell"]),
         Paragraph(f"(1 − horas/umbral_clima) × 10", styles["table_cell"]),
         Paragraph(f"Umbral: maíz {CONFIG['maiz']['umbral_clima']}h | soja {CONFIG['soja']['umbral_clima']}h | trigo {CONFIG['trigo']['umbral_clima']}h", styles["table_cell"])],
        [Paragraph("Alerta NDVI bajo", styles["table_cell"]), Paragraph("GEE NDVI crítico", styles["table_cell"]), Paragraph("NDVI < umbral cultivo (maíz:0.50, soja:0.45, trigo:0.40)", styles["table_cell"]), Paragraph("No excluye el dato, es bandera de campo", styles["table_cell"])],
        [Paragraph("Área del lote", styles["table_cell"]), Paragraph("Shapefile + UTM dinámico", styles["table_cell"]), Paragraph(f"Reprojeción EPSG:{res['epsg_utm']}", styles["table_cell"]), Paragraph("Exacto para polígonos <100km²", styles["table_cell"])],
    ]
    mt = Table(met_tabla, colWidths=[3.2*cm, 3.5*cm, 4.8*cm, 3*cm])
    mt.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,0), C_PRIMARY),
                             ("ROWBACKGROUNDS", (0,1), (-1,-1), [white, C_ACCENT_BG]),
                             ("GRID", (0,0), (-1,-1), 0.3, C_BORDER),
                             ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
                             ("TOPPADDING", (0,0), (-1,-1), 5), ("BOTTOMPADDING", (0,0), (-1,-1), 5),
                             ("FONTSIZE", (0,0), (-1,-1), 7.5)]))
    story.append(mt)
    story.append(Spacer(1, 0.5*cm))
    story.append(Paragraph("Log de procesamiento (trazabilidad completa)", styles["h3"]))
    for entrada in res.get("log", []):
        story.append(Paragraph(entrada, styles["log"]))
    story.append(Spacer(1, 0.8*cm))
    story.append(HRFlowable(width="100%", thickness=0.3, color=C_BORDER, spaceAfter=4))
    story.append(Paragraph(
        f"AgroIA Report v{VERSION} — Generado: {fecha_str} — "
        f"Fuentes: NASA POWER, GEE Copernicus/S2_SR_HARMONIZED, IsolationForest (sklearn) — "
        f"Las estimaciones de estrés térmico se basan en bibliografía de {res['conf']['biblio']} "
        f"y no constituyen garantía de resultado.", styles["meta"]))

    doc.build(story, onFirstPage=decorator, onLaterPages=decorator)

    writer = PdfWriter()
    for path in [cover_path, body_path]:
        reader = PdfReader(path)
        for page in reader.pages:
            writer.add_page(page)
    with open(output_path, "wb") as f:
        writer.write(f)
    os.unlink(cover_path)
    os.unlink(body_path)
    print(f"✅ PDF generado: {output_path}")
    return output_path


# ============================================================================
# EJECUCIÓN — BLOQUE FINAL UNIFICADO v2.5
# ============================================================================

from google.colab import files

# 1. Inicializar GEE
init_gee(project_id="applied-oxygen-459415-e2")

# 2. Rango de años dinámico
año_actual     = datetime.now().year
years_dinamicos = list(range(año_actual - 5, año_actual + 1))

# 3. Subir Shapefile
print("📂 Subí los 4 archivos del lote: .shp, .shx, .dbf, .prj")
uploaded = files.upload()
shp_path = [f for f in uploaded.keys() if f.endswith('.shp')][0]

# 4. Correr pipeline
res = run_agroia_pipeline(shp_path, cultivo="maiz", years=years_dinamicos)
verificar_resultado(res)

# ════════════════════════════════════════════════════════════════
# ★ EDITÁ SOLO ESTA LÍNEA — nombre unificado para PDF, HTML, CSV y RAG
# Pedir al usuario que ingrese el nombre del lote
nombre_lote = input("Ingrese el nombre del lote: ")
# ════════════════════════════════════════════════════════════════

# 5. Zonificación y conteo Zona C
z_gdf        = None
zona_c_count = 0
if res["es_variable"] and res["zonas_gdf"] is not None:
    print("🎨 Renderizando ambientes...")
    z_gdf = res["zonas_gdf"].copy()
    z_gdf["geometry"] = z_gdf.geometry.buffer(0.00015).centroid
    zona_c_count = len(z_gdf[z_gdf['zona'] == 'C'])

# 6. Historial con scores completos (para RAG y Streamlit)
umbral_clima_hist = res['conf'].get('umbral_clima', 40)
años_hist         = sorted(res["ndvi_historico"].keys())
historial_completo = []
for a in años_hist:
    ndvi  = res["ndvi_historico"][a]
    horas = res["horas_calor_hist"].get(a, 0)
    previos = [res["ndvi_historico"][y] for y in años_hist
               if y < a and y not in res.get("anos_excluidos", [])]
    vigor = float(np.clip(ndvi / 0.9, 0, 1) * 40)
    clima = float(np.clip(1 - horas / umbral_clima_hist, 0, 1) * 10)
    if len(previos) >= 2:
        cv_p = np.std(previos) / np.mean(previos) if np.mean(previos) > 0 else 0
        est  = float(np.clip(1 - cv_p / 0.45, 0, 1) * 30)
    else:
        est = float(np.clip(15.0 + (ndvi - 0.6) * 10, 10, 25))
    if len(previos) >= 4:
        iso    = IsolationForest(contamination=0.2, random_state=42)
        labels = iso.fit_predict(np.array(previos).reshape(-1, 1))
        limp   = float(np.clip(1 - (labels==-1).sum()/len(labels)/0.40, 0, 1) * 20)
    else:
        limp = float(np.clip(10.0 + (ndvi - 0.6) * 5, 8, 15))
    historial_completo.append({
        "anio": int(a), "ndvi_critico": round(ndvi, 4), "horas_calor": round(horas, 1),
        "score_total": int(round(vigor + est + limp + clima)),
        "score_vigor": round(vigor, 1), "score_estabilidad": round(est, 1),
        "score_limpieza": round(limp, 1), "score_clima": round(clima, 1),
        "cultivo": res.get("cultivo", "maiz"),
        "valido_para_score": a not in res.get("anos_excluidos", []),
        "superficie_ha": res.get("hectareas", 0),
        "cv_espacial": res.get("cv", 0),
        "zonificacion_activa": res.get("es_variable", False),
        "puntos_zona_c": zona_c_count if a == max(años_hist) else 0,
    })

# 7. Generar PDF
pdf_path = build_report(res, lote_id=nombre_lote,
                        output_path=f"AgroIA_{nombre_lote}.pdf")

# 8. Generar HTML + CSV Zona C
mapa_path, csv_path = generar_mapa_offline(
    lote_gdf=res["lote_gdf"], cultivo=res["cultivo"],
    score=res["score"]["total"], conf=res["conf"],
    zonas_gdf=z_gdf, output_path=f"Mapa_{nombre_lote}.html"
)

# 9. Descargar todo
print("\n🚀 Descargando entregables...")
files.download(pdf_path)
files.download(mapa_path)
if csv_path:
    print(f"📥 Descargando CSV Zona C...")
    files.download(csv_path)

print(f"\n{'='*55}")
print(f"  ✅  nombre_lote    = '{nombre_lote}'")
print(f"  📊  Historial      = {len(historial_completo)} campañas con scores")
print(f"  🗺️   Zona C         = {zona_c_count} puntos críticos")
print(f"  → Ejecutá ahora las celdas del notebook RAG (Celda 5)")
print(f"{'='*55}")


✓ GEE inicializado correctamente.
📂 Subí los 4 archivos del lote: .shp, .shx, .dbf, .prj


Saving MAIZ-CLAP.dbf to MAIZ-CLAP (3).dbf
Saving MAIZ-CLAP.shp to MAIZ-CLAP (3).shp
Saving MAIZ-CLAP.shx to MAIZ-CLAP (3).shx
Saving MAIZ-CLAP.cpg to MAIZ-CLAP (3).cpg
Saving MAIZ-CLAP.prj to MAIZ-CLAP (3).prj

🔍 Validando shapefile...
✓ Shapefile valido: 1.21 ha | Polygon | EPSG:32615
✓ Lote: 1.21 ha | EPSG:32615 | Centroide: 14.8758, -91.5378

🌡️  Descargando NASA POWER (6 años)...
   ✓ NASA 2021: 0.0h
   ✓ NASA 2022: 0.0h
   ✓ NASA 2023: 0.0h
   ✓ NASA 2024: 0.0h
   ✓ NASA 2025: 0.0h
   ✓ NASA 2026: 0.0h

🛰️  Procesando GEE Sentinel-2 (mes critico: 1)...
   ✓ 2021/01: NDVI=0.358 plausible.
   ⚠ 2022/01: NDVI=0.233 bajo mínimo.
   ✓ 2023/01: NDVI=0.285 plausible.
   ⚠ 2024/01: NDVI=0.231 bajo mínimo.
   ✓ 2025/01: NDVI=0.342 plausible.
   ✓ 2026/01: NDVI=0.273 plausible.

📊 Analizando ambientes espaciales...
   ✓ CV=0.280 → Zonificación A/B/C activada
   ⚠ Años excluidos del Score: [2022, 2024]

  ✅  SCORE AGROIA: 52/100
      Vigor         12.1 / 40
      Estabilidad   22.3 / 30
   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Descargando CSV Zona C...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  ✅  nombre_lote    = 'Lote CLAP 2026'
  📊  Historial      = 4 campañas con scores
  🗺️   Zona C         = 76 puntos críticos
  → Ejecutá ahora las celdas del notebook RAG (Celda 5)


---
## ⚙️ Celda 1 — Dependencias

In [ ]:
# requests normalmente ya está en Colab.
# Si no, esto lo instala silenciosamente.
try:
    import requests
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'requests', '-q'], check=True)
    import requests

import json
import pandas as pd
from datetime import date

print('✅ Dependencias listas')

✅ Dependencias listas


---
## ⚙️ Celda 2A — Configuración (servidor local)

Editá `RAG_API_URL` y `RAG_SECRET_KEY` con los valores de tu instalación.
Estos deben coincidir con los valores en `config/.env` del servidor AgroIA RAG.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  EDITÁ ESTOS DOS VALORES ANTES DE CORRER
# ════════════════════════════════════════════════════════════════

RAG_API_URL    = 'http://localhost:8000'    # URL del servidor AgroIA RAG
RAG_SECRET_KEY = 'mi_clave_super_secreta'  # Valor de INGESTA_SECRET_KEY en config/.env

# ── Modo debug ───────────────────────────────────────────────────
# True  → inserción síncrona, ves el ID asignado en Colab (recomendado para verificar)
# False → inserción en background, más rápido (para producción)
RAG_DEBUG_MODE = True

print(f'📡 API configurada: {RAG_API_URL}')
print(f'🔑 Clave cargada:   {RAG_SECRET_KEY[:6]}...')
print(f'🐛 Modo debug:      {RAG_DEBUG_MODE}')

📡 API configurada: http://localhost:8000
🔑 Clave cargada:   mi_cla...
🐛 Modo debug:      True


---
## ⚙️ Celda 2B — Configuración con ngrok (servidor remoto)

Usá esta celda **en lugar de la 2A** si el servidor corre en otra máquina y necesitás acceder por internet.

In [ ]:
# ── Opción A: pegar la URL de ngrok manualmente ──────────────────────────────
# 1. En la terminal del servidor: ngrok http 8000
# 2. Copiá la URL que aparece (ej: https://xxxx.ngrok-free.app)
# 3. Pegala abajo

RAG_API_URL    = 'https://blooper-goofiness-habitual.ngrok-free.dev'  # ← pegá tu URL ngrok
RAG_SECRET_KEY = 'mi_clave_super_secreta'
RAG_DEBUG_MODE = True

# ── Opción B: crear el tunnel desde Colab con pyngrok ───────────────────────
# (requiere cuenta en ngrok.com, token gratuito)
#
# !pip install pyngrok -q
# from pyngrok import ngrok
# ngrok.set_auth_token('TU_AUTHTOKEN_DE_NGROK')   # https://dashboard.ngrok.com
# tunnel = ngrok.connect(8000)
# RAG_API_URL    = tunnel.public_url
# RAG_SECRET_KEY = 'mi_clave_super_secreta'
# RAG_DEBUG_MODE = True
# print(f'🌐 Tunnel activo: {RAG_API_URL}')

print('Descomentá una de las opciones y ejecutá esta celda.')

Descomentá una de las opciones y ejecutá esta celda.


---
## 🔧 Celda 3 — Funciones de integración RAG

Ejecutá esta celda una vez para cargar todas las funciones. No necesitás editarla.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  AgroIA RAG — Módulo de integración inline v2.4
#  Usa los nombres de variables reales del pipeline AGROIA_EXTENSIVOS.
#  No modificar. Solo ejecutar.
# ════════════════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import io, glob
from datetime import date

# ── pypdf para adjuntar la página al PDF del pipeline ────────────────────────
try:
    from pypdf import PdfWriter, PdfReader
    _PYPDF_OK = True
except ImportError:
    _PYPDF_OK = False


def _serializar_historial(historial_raw, fallback_estabilidad=0.0, fallback_limpieza=0.0):
    """
    Convierte historial_completo (lista de dicts o DataFrame) al formato
    esperado por la API del RAG.

    Si una campaña histórica no tiene scores calculados (solo ndvi_critico
    y horas_calor), los recalcula con la misma fórmula del pipeline:
      • score_vigor     = clip(ndvi / 0.9, 0, 1) × 40
      • score_clima     = clip(1 - horas / 20, 0, 1) × 10
      • score_estabilidad / score_limpieza → usa fallback del año actual
        (estas métricas dependen de la serie espacial completa del lote)
    """
    if historial_raw is None:
        return []

    # Aceptar tanto lista de dicts como DataFrame
    import pandas as pd
    if isinstance(historial_raw, pd.DataFrame):
        filas_raw = historial_raw.to_dict('records')
    else:
        filas_raw = list(historial_raw)

    # Mapeo de nombres del pipeline → nombres del payload RAG
    alias = {
        'anio':              'anio',
        'año':               'anio',
        'año_campana':       'anio',
        'ndvi_critico':      'ndvi_critico',
        'ndvi_medio':        'ndvi_critico',
        'horas_calor':       'horas_calor',
        'horas_estres':      'horas_calor',
        'score_total':       'score_total',
        'score_vigor':       'score_vigor',
        'score_estabilidad': 'score_estabilidad',
        'score_limpieza':    'score_limpieza',
        'score_clima':       'score_clima',
        'cultivo':           'cultivo',
        'valido_para_score': 'valido_para_score',
        'valido':            'valido_para_score',
        'superficie_ha':     'superficie_ha',
        'cv_espacial':       'cv_espacial',
        'cv':                'cv_espacial',
        'zonificacion_activa': 'zonificacion_activa',
        'zona_activa':       'zonificacion_activa',
        'es_variable':       'zonificacion_activa',
        'puntos_zona_c':     'puntos_zona_c',
    }

    filas = []
    for raw in filas_raw:
        entrada = {}
        for k, v in raw.items():
            destino = alias.get(k, k)
            # Convertir tipos numpy a Python nativos
            if hasattr(v, 'item'):
                v = v.item()
            elif hasattr(v, 'tolist'):
                v = v.tolist()
            entrada[destino] = v

        if 'anio' not in entrada:
            print(f'⚠️  Fila sin año — omitida: {list(raw.keys())}')
            continue
        entrada['anio'] = int(entrada['anio'])

        # ── Calcular scores faltantes ─────────────────────────────────────────
        ndvi_h  = float(entrada.get('ndvi_critico') or 0)
        horas_h = float(entrada.get('horas_calor') or 0)

        if not entrada.get('score_vigor'):
            entrada['score_vigor'] = round(min(max(ndvi_h / 0.9, 0.0), 1.0) * 40, 2)
        if not entrada.get('score_clima'):
            entrada['score_clima'] = round(max(0.0, 1.0 - horas_h / 20.0) * 10, 2)
        if not entrada.get('score_estabilidad'):
            entrada['score_estabilidad'] = round(float(fallback_estabilidad), 2)
        if not entrada.get('score_limpieza'):
            entrada['score_limpieza'] = round(float(fallback_limpieza), 2)
        if not entrada.get('score_total'):
            entrada['score_total'] = round(
                entrada['score_vigor'] + entrada['score_estabilidad'] +
                entrada['score_limpieza'] + entrada['score_clima']
            )
        # ─────────────────────────────────────────────────────────────────────
        filas.append(entrada)

    return sorted(filas, key=lambda r: r['anio'])


def _construir_contenido_tecnico(lote_id, cultivo, superficie_ha,
                                  ndvi_critico, horas_calor,
                                  score_total, score_vigor, score_estabilidad,
                                  score_limpieza, score_clima,
                                  cv_espacial, zona_activa, historial_filas):
    zona_str = 'Activa (K-Means)' if zona_activa else 'Homogéneo (CV ≤ 0.05)'
    lineas = []
    for h in historial_filas:
        estado = '✓' if h.get('valido_para_score', True) else 'EXCLUIDO'
        lineas.append(
            f"  {h['anio']}: NDVI {h.get('ndvi_critico',0):.3f} | "
            f"Estres {h.get('horas_calor',0):.1f}h | "
            f"Score {h.get('score_total',0)}/100 "
            f"(V:{h.get('score_vigor',0):.1f} E:{h.get('score_estabilidad',0):.1f} "
            f"L:{h.get('score_limpieza',0):.1f} C:{h.get('score_clima',0):.1f}) [{estado}]"
        )
    hist_str = '\n'.join(lineas) if lineas else '  Sin historial registrado.'
    return (
        f"AgroIA {lote_id.upper()} | {cultivo.upper()} | {superficie_ha:.2f} ha\n"
        f"Score consolidado: {score_total}/100\n"
        f"Componentes: Vigor {score_vigor:.1f}/40 | Estabilidad {score_estabilidad:.1f}/30 "
        f"| Limpieza {score_limpieza:.1f}/20 | Clima {score_clima:.1f}/10\n"
        f"NDVI critico actual: {ndvi_critico:.3f} | Estres termico: {horas_calor:.1f}h\n"
        f"CV espacial: {cv_espacial:.3f} | Variabilidad: {zona_str}\n"
        f"\nHistorial por campana:\n{hist_str}"
    )


def construir_payload_v2(lote_id, cultivo, superficie_ha, fecha_analisis,
                          ndvi_critico, horas_calor,
                          score_total, score_vigor, score_estabilidad,
                          score_limpieza, score_clima,
                          cv_espacial, zona_activa, puntos_zona_c,
                          historial_raw=None, version_agroia='2.4'):
    historial_filas = _serializar_historial(
        historial_raw,
        fallback_estabilidad=score_estabilidad,
        fallback_limpieza=score_limpieza
    )
    contenido = _construir_contenido_tecnico(
        lote_id, cultivo, superficie_ha, ndvi_critico, horas_calor,
        score_total, score_vigor, score_estabilidad, score_limpieza, score_clima,
        cv_espacial, zona_activa, historial_filas)
    return {
        'lote_id':           lote_id,
        'fecha':             str(fecha_analisis),
        'ndvi_promedio':     round(float(ndvi_critico), 4),
        'gdd_acumulados':    round(float(horas_calor), 1),
        'contenido_tecnico': contenido,
        'metadata': {
            'cultivo':             cultivo,
            'superficie_ha':       round(float(superficie_ha), 2),
            'score_total':         int(score_total),
            'score_desglose': {
                'vigor':       round(float(score_vigor), 2),
                'estabilidad': round(float(score_estabilidad), 2),
                'limpieza':    round(float(score_limpieza), 2),
                'clima':       round(float(score_clima), 2),
            },
            'cv_espacial':         round(float(cv_espacial), 4),
            'zonificacion_activa': bool(zona_activa),
            'puntos_zona_c':       int(puntos_zona_c),
            'version_agroia':      version_agroia,
        },
        'historial_años': historial_filas,
    }


def verificar_conexion_rag(url=None):
    _url = (url or RAG_API_URL).rstrip('/')
    try:
        r = requests.get(f'{_url}/health', timeout=5)
        if r.status_code == 200:
            d = r.json()
            print('✅  API AgroIA RAG disponible')
            print(f'    DB: {d.get("db")} | Modelo: {d.get("embedding_model")} | v{d.get("version")}')
            return True
        print(f'⚠️  API responde {r.status_code}: {r.text}')
        return False
    except requests.exceptions.ConnectionError:
        print(f'❌  Sin conexión a {_url}')
        print('    → ¿Está corriendo? python start.py --api')
        print('    → ¿Colab remoto? Configurá ngrok en Celda 2B.')
        return False
    except Exception as e:
        print(f'❌  Error inesperado: {e}')
        return False


def enviar_al_rag(lote_id, cultivo, superficie_ha, fecha_analisis,
                   ndvi_critico, horas_calor,
                   score_total, score_vigor, score_estabilidad,
                   score_limpieza, score_clima,
                   cv_espacial, zona_activa, puntos_zona_c,
                   historial_raw=None, version_agroia='2.4'):
    payload = construir_payload_v2(
        lote_id=lote_id, cultivo=cultivo, superficie_ha=superficie_ha,
        fecha_analisis=fecha_analisis, ndvi_critico=ndvi_critico, horas_calor=horas_calor,
        score_total=score_total, score_vigor=score_vigor, score_estabilidad=score_estabilidad,
        score_limpieza=score_limpieza, score_clima=score_clima,
        cv_espacial=cv_espacial, zona_activa=zona_activa, puntos_zona_c=puntos_zona_c,
        historial_raw=historial_raw, version_agroia=version_agroia)

    n_hist = len(payload['historial_años'])
    print('━' * 58)
    print(f'📡  Enviando a:  {RAG_API_URL}')
    print(f'🌾  Lote:        {lote_id}')
    print(f'📅  Fecha:       {fecha_analisis}')
    print(f'🌿  NDVI:        {ndvi_critico:.3f}  |  Estrés: {horas_calor:.1f}h')
    print(f'🏆  Score:       {score_total}/100  (V:{score_vigor:.1f} E:{score_estabilidad:.1f} L:{score_limpieza:.1f} C:{score_clima:.1f})')
    print(f'📊  Historial:   {n_hist} campaña(s) con scores por año')
    print('━' * 58)

    endpoint = f'{RAG_API_URL.rstrip("/")}/ingesta/debug' if RAG_DEBUG_MODE \
               else f'{RAG_API_URL.rstrip("/")}/ingesta'
    try:
        resp = requests.post(endpoint, json=payload,
                             headers={'Authorization': f'Bearer {RAG_SECRET_KEY}',
                                      'Content-Type': 'application/json'},
                             timeout=60)
    except requests.exceptions.Timeout:
        print('⏱️  Timeout — intentá con RAG_DEBUG_MODE = False.')
        raise
    except requests.exceptions.ConnectionError:
        print('❌  Conexión rechazada.')
        raise

    if resp.status_code in (200, 201, 202):
        data = resp.json()
        if RAG_DEBUG_MODE:
            print(f'✅  Guardado en RAG | ID: {data.get("id", "N/D")}')
        else:
            print('✅  Aceptado (procesando en background)')
        print(f'    lote_id: {data.get("lote_id")}  |  fecha: {data.get("fecha", fecha_analisis)}')
        print(f'\n🌐 Disponible en Streamlit y Telegram.')
        return data
    elif resp.status_code == 401:
        print('❌  Clave incorrecta (401). Verificá RAG_SECRET_KEY.')
        raise ValueError('Unauthorized')
    elif resp.status_code == 422:
        print(f'❌  Payload inválido (422): {resp.text}')
        raise ValueError(resp.text)
    else:
        print(f'❌  Error {resp.status_code}: {resp.text}')
        resp.raise_for_status()


def listar_lotes_rag(url=None):
    _url = (url or RAG_API_URL).rstrip('/')
    try:
        r = requests.get(f'{_url}/lotes', timeout=10)
        r.raise_for_status()
        data  = r.json()
        lotes = data.get('lotes', [])
        print(f'📋  Lotes en el RAG: {data.get("total", 0)}')
        print(f'    {"LOTE":<32} {"CULTIVO":<8} {"SCORE":>5}  {"NDVI":>6}  FECHA')
        print('    ' + '─' * 62)
        for l in lotes:
            print(f'    {l["lote_id"]:<32} {(l.get("cultivo") or "N/D"):<8} '
                  f'{str(l.get("score_total","N/D")):>5}  {l.get("ndvi_promedio",0):.3f}  '
                  f'{l.get("fecha") or "-"}')
        return [l['lote_id'] for l in lotes]
    except Exception as e:
        print(f'❌  No se pudo listar lotes: {e}')
        return []


def verificar_lote_rag(lote_id, url=None):
    _url = (url or RAG_API_URL).rstrip('/')
    try:
        r = requests.get(f'{_url}/lotes/{lote_id}', timeout=10)
        if r.status_code == 404:
            print(f'⚠️  Lote "{lote_id}" no encontrado. Si enviaste en modo background, esperá unos segundos.')
            return False
        r.raise_for_status()
        data    = r.json()
        informe = data.get('informe', {})
        hist    = data.get('historial', [])
        print(f'✅  Lote "{lote_id}" confirmado en el RAG')
        print(f'    Score: {informe.get("score_total","N/D")}/100 | NDVI: {informe.get("ndvi_promedio","N/D")} | Cultivo: {informe.get("cultivo","N/D")}')
        print(f'    Historial: {len(hist)} campaña(s)')
        if hist:
            print(f'    {"AÑO":<6} {"NDVI":>7}  {"ESTRES":>8}h  {"SCORE":>5}  VIGOR  ESTAB  LIMP  CLIM')
            print('    ' + '─' * 58)
            for h in hist:
                valido = '' if h.get('valido_para_score', True) else ' ✗EXCL'
                print(f'    {h["anio"]:<6} {h.get("ndvi_critico",0):>7.3f}  '
                      f'{h.get("horas_calor",0):>8.1f}  '
                      f'{h.get("score_total",0):>5}/100  '
                      f'{h.get("score_vigor",0):>5.1f}  '
                      f'{h.get("score_estabilidad",0):>5.1f}  '
                      f'{h.get("score_limpieza",0):>4.1f}  '
                      f'{h.get("score_clima",0):>4.1f}{valido}')
        return True
    except Exception as e:
        print(f'❌  Error: {e}')
        return False


print('✅  Funciones RAG cargadas (variables reales del pipeline AgroIA v2.4)')

✅  Funciones RAG cargadas (variables reales del pipeline AgroIA v2.4)


---
## 🔌 Celda 4 — Verificar conexión con el servidor RAG

Corré esta celda antes de enviar. Si falla, revisá la URL y que el servidor esté levantado.

In [ ]:
verificar_conexion_rag()

✅  API AgroIA RAG disponible
    DB: agri_db | Modelo: nomic-embed-text | v2.0


True

---
## 🚀 Celda 5 — Enviar al RAG

Esta celda usa las variables que ya calculó el pipeline AgroIA v2.4.

**Revisá los nombres de variables** en el bloque `# MAPEO` y ajustá si en tu notebook
se llaman diferente (por ejemplo `horas_estres` en lugar de `horas_calor`).

In [ ]:
# ════════════════════════════════════════════════════════════════
#  MAPEO AUTOMÁTICO — variables del pipeline (no tocar)
# ════════════════════════════════════════════════════════════════
fecha_analisis = date.today()
_lote_id           = nombre_lote
_cultivo           = res['cultivo']
_superficie_ha     = res['hectareas']
_fecha_analisis    = fecha_analisis
_ndvi_critico      = res['ndvi_critico_actual']
_horas_calor       = res['horas_calor_actual']
_score_total       = res['score']['total']
_score_vigor       = res['score']['vigor']
_score_estabilidad = res['score']['estabilidad']
_score_limpieza    = res['score']['limpieza']
_score_clima       = res['score']['clima']
_cv_espacial       = res['cv']
_zona_activa       = res['es_variable']
try:
    _puntos_zona_c = len(zona_c_gdf) if res['es_variable'] else 0
except NameError:
    _puntos_zona_c = 0
_historial_raw     = historial_completo

# ── Enviar ──────────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════
#  Celda 5 — Enviar al RAG
#  Usa nombre_lote, historial_completo y zona_c_count definidos
#  en el bloque final del pipeline base (AGROIA_EXTENSIVOS).
#  No tocar nada aquí.
# ════════════════════════════════════════════════════════════════

resultado = enviar_al_rag(
    lote_id           = nombre_lote,
    cultivo           = res['cultivo'],
    superficie_ha     = res['hectareas'],
    fecha_analisis    = fecha_analisis,
    ndvi_critico      = res['ndvi_critico_actual'],
    horas_calor       = res['horas_calor_actual'],
    score_total       = res['score']['total'],
    score_vigor       = res['score']['vigor'],
    score_estabilidad = res['score']['estabilidad'],
    score_limpieza    = res['score']['limpieza'],
    score_clima       = res['score']['clima'],
    cv_espacial       = res['cv'],
    zona_activa       = res['es_variable'],
    puntos_zona_c     = zona_c_count,
    historial_raw     = historial_completo,
    version_agroia    = '2.4',
)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📡  Enviando a:  https://blooper-goofiness-habitual.ngrok-free.dev
🌾  Lote:        Lote CLAP 2026
📅  Fecha:       2026-04-20
🌿  NDVI:        0.273  |  Estrés: 0.0h
🏆  Score:       52/100  (V:12.1 E:22.3 L:7.5 C:10.0)
📊  Historial:   4 campaña(s) con scores por año
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅  Guardado en RAG | ID: 95
    lote_id: Lote CLAP 2026  |  fecha: 2026-04-20

🌐 Disponible en Streamlit y Telegram.


---
## ✅ Celda 6 — Verificación final

Confirma que el lote quedó registrado correctamente y lista todos los lotes disponibles en el RAG.

In [ ]:
# ── Listar todos los lotes en el RAG ────────────
listar_lotes_rag()

print()

# ── Verificar el lote recién enviado ────────────
verificar_lote_rag(_lote_id)

📋  Lotes en el RAG: 3
    LOTE                             CULTIVO  SCORE    NDVI  FECHA
    ──────────────────────────────────────────────────────────────
    Lote Circual - Maiz 123          maiz        69  0.910  2026-04-20
    Lote CLAP 2026                   maiz        52  0.273  2026-04-20
    Lote GIGANTE - Maiz 231          maiz        80  0.827  2026-04-20

✅  Lote "Lote CLAP 2026" confirmado en el RAG
    Score: 52/100 | NDVI: 0.2729 | Cultivo: maiz
    Historial: 4 campaña(s)
    AÑO       NDVI    ESTRESh  SCORE  VIGOR  ESTAB  LIMP  CLIM
    ──────────────────────────────────────────────────────────
    2021     0.358       0.0     47/100   15.9   12.6   8.8  10.0
    2023     0.285       0.0     43/100   12.7   11.9   8.4  10.0
    2025     0.342       0.0     56/100   15.2   22.4   8.7  10.0
    2026     0.273       0.0     54/100   12.1   23.6   8.4  10.0


True

---
## ℹ️ Referencia rápida — Columnas esperadas en `historial_df`

El módulo acepta cualquiera de estos nombres de columna (son equivalentes):

| Dato | Nombres aceptados |
|---|---|
| Año de campaña | `anio`, `año`, `año_campana`, `year` |
| NDVI crítico | `ndvi_critico`, `ndvi_medio`, `ndvi_actual` |
| Horas de estrés | `horas_calor`, `horas_estres`, `horas_sobre_umbral` |
| Score total | `score_total` |
| Componentes | `score_vigor`, `score_estabilidad`, `score_limpieza`, `score_clima` |
| Válido para score | `valido_para_score`, `valido` |
| CV espacial | `cv_espacial`, `cv` |
| Zonificación | `zonificacion_activa`, `zona_activa` |

Si alguna columna no está en el DataFrame, se usa el valor del año actual como fallback.